# Chapitre 2 - Exercices 2.1 à 2.5

Ce notebook traite les cinq premiers exercices du chapitre 2 :

- **2.1** : classer au plus grand $\hat y_k$ revient à choisir la cible la plus proche ;
- **2.2** : calcul de la frontière de Bayes de la simulation mixture ;
- **2.3** : distance médiane au plus proche voisin, éq. (2.24) ;
- **2.4** : en grande dimension, les points tests sont « au bord » de l'échantillon ;
- **2.5** : erreur de prédiction du modèle linéaire, éq. (2.27) et (2.28).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, special
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl.plotting import decision_grid
from esl import datasets
from esl import ch02_supervise as c2

set_style()
rng = np.random.RandomState(0)

## Exercice 2.1 : Plus grande composante et cible la plus proche

**Énoncé (reformulé).** On a $K$ classes ; la classe $k$ est associée à une **cible**
$t_k \in \mathbb R^K$, vecteur dont toutes les composantes sont nulles sauf la $k$-ième
qui vaut 1. Pour une prédiction $\hat y \in \mathbb R^K$ dont les composantes ont pour
somme 1, montrer que classer selon la plus grande composante de $\hat y$ revient à
choisir la cible la plus proche, c'est-à-dire la classe $k$ minimisant $\|t_k - \hat y\|$.

**Contexte.** C'est la situation de la régression sur variables indicatrices : on code
chaque classe par le vecteur $t_k$, on ajuste chacune des $K$ coordonnées par
moindres carrés, et l'on obtient pour un nouveau $x$ un vecteur $\hat y(x)$. Lorsque le
modèle contient une constante, les composantes de $\hat y$ somment à 1 (chapitre 4).

**Solution.** Développons le carré de la distance (le minimum de $\|t_k - \hat y\|$ et
celui de $\|t_k - \hat y\|^2$ sont atteints au même $k$, car $u \mapsto u^2$ est
croissante sur $\mathbb R_+$) :
$$\|t_k - \hat y\|^2 = \|t_k\|^2 - 2\,t_k^T\hat y + \|\hat y\|^2.$$
Or $\|t_k\|^2 = 1$ (une seule composante non nulle, égale à 1) et $t_k^T\hat y = \hat y_k$
(le produit scalaire avec $t_k$ extrait la $k$-ième composante). Donc
$$\|t_k - \hat y\|^2 = 1 - 2\hat y_k + \|\hat y\|^2.$$

Le terme $1 + \|\hat y\|^2$ ne dépend pas de $k$. Par conséquent
$$\arg\min_k \|t_k - \hat y\|^2 = \arg\min_k\,(-2\hat y_k) = \arg\max_k \hat y_k. \qquad\blacksquare$$

> **Remarque.** L'hypothèse « les $\hat y_k$ somment à 1 » n'a en fait **pas servi** :
> l'équivalence est vraie pour tout $\hat y \in \mathbb R^K$. Elle sert seulement à
> interpréter $\hat y$ comme une estimation des probabilités des classes.

**Vérification numérique** sur 10 000 vecteurs aléatoires de $\mathbb R^5$, normalisés
(somme 1) ou non.

In [ ]:
K = 5
T = np.eye(K)                       # les cibles t_k sont les lignes de l'identité
Y = rng.normal(size=(10000, K))
for name, Yhat in [("quelconques", Y), ("somme = 1", Y - Y.mean(1, keepdims=True) + 1 / K)]:
    dist = ((Yhat[:, None, :] - T[None, :, :]) ** 2).sum(-1)    # (10000, K)
    agree = np.mean(dist.argmin(1) == Yhat.argmax(1))
    print("%-12s : accord argmin distance / argmax composante = %.4f" % (name, agree))

Visualisation pour $K = 3$ sur le plan $\{\hat y_1 + \hat y_2 + \hat y_3 = 1\}$ (paramétré
par $(\hat y_1, \hat y_2)$) : les régions « cible la plus proche » et « plus grande
composante » coïncident.

In [ ]:
u, v, P = decision_grid((-0.5, 1.5), (-0.5, 1.5), n=300)
Yh = np.c_[P, 1 - P.sum(1)]
lab_max = Yh.argmax(1).reshape(u.shape)
lab_dist = ((Yh[:, None, :] - np.eye(3)[None]) ** 2).sum(-1).argmin(1).reshape(u.shape)
from matplotlib.colors import ListedColormap
cm3 = ListedColormap(COLORS[:3])
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, lab, name in zip(axes, [lab_max, lab_dist], ["plus grande composante", "cible la plus proche"]):
    ax.pcolormesh(u, v, lab, cmap=cm3, shading="auto", alpha=0.5)
    ax.plot([1, 0, 0], [0, 1, 0], "k*", ms=12)
    ax.set_xlabel("$\\hat y_1$")
    ax.set_ylabel("$\\hat y_2$")
    ax.set_title(name)
    ax.grid(False)
for k in range(3):
    axes[0].plot([], [], "s", color=COLORS[k], alpha=0.6, label="classe %d" % (k + 1))
axes[0].legend(loc="upper right")
plt.show()

> **À retenir.** $\|t_k - \hat y\|^2 = 1 - 2\hat y_k + \|\hat y\|^2$ : choisir la cible la
> plus proche, c'est choisir la plus grande composante de $\hat y$.

## Exercice 2.2 : Calcul de la frontière de Bayes de la simulation mixture

**Énoncé (reformulé).** Expliquer comment calculer la frontière de décision de Bayes
de l'exemple simulé de la figure 2.5.

**Rappels.** Le modèle génératif (section 2.3.3) est :

- classe BLEU : $X \sim \frac1{10}\sum_{k=1}^{10}\mathcal N(m_k, \mathbf I/5)$, centres $m_k$ tirés selon $\mathcal N((1,0)^T, \mathbf I)$ ;
- classe ORANGE : $X \sim \frac1{10}\sum_{k=1}^{10}\mathcal N(n_k, \mathbf I/5)$, centres $n_k$ tirés selon $\mathcal N((0,1)^T, \mathbf I)$ ;
- les deux classes ont la même probabilité a priori $\pi_B = \pi_O = 1/2$.

Le classifieur de Bayes (éq. (2.23)) choisit la classe de plus grande probabilité a
posteriori ; la frontière est l'ensemble des $x$ où les deux probabilités a posteriori
sont égales.

**Étape 1 : densités des classes.** La densité de $\mathcal N(m, \sigma^2\mathbf I_2)$, avec $\sigma^2 = 1/5$, est
$$\varphi_m(x) = \frac{1}{2\pi\sigma^2}\exp\Big(-\frac{\|x - m\|^2}{2\sigma^2}\Big) = \frac{5}{2\pi}\exp\Big(-\frac52\|x - m\|^2\Big).$$
Donc $p_B(x) = \frac1{10}\sum_k\varphi_{m_k}(x)$ et $p_O(x) = \frac1{10}\sum_k\varphi_{n_k}(x)$.

**Étape 2 : probabilités a posteriori.** Par la formule de Bayes,
$$\Pr(\text{ORANGE}\mid X = x) = \frac{\pi_O\,p_O(x)}{\pi_B\,p_B(x) + \pi_O\,p_O(x)} = \frac{p_O(x)}{p_B(x) + p_O(x)}.$$

**Étape 3 : la frontière.** $\Pr(\text{ORANGE}\mid x) = \Pr(\text{BLEU}\mid x)$ équivaut à
$p_O(x) = p_B(x)$, c'est-à-dire (en simplifiant les constantes communes $\frac{1}{10}\cdot\frac{5}{2\pi}$) :
$$\sum_{k=1}^{10}\exp\Big(-\frac52\|x - n_k\|^2\Big) = \sum_{k=1}^{10}\exp\Big(-\frac52\|x - m_k\|^2\Big).$$
C'est une courbe implicite $\{x : \delta(x) = 0\}$, avec
$\delta(x) = \log\sum_k e^{-\frac52\|x-n_k\|^2} - \log\sum_k e^{-\frac52\|x-m_k\|^2}$. Elle n'a
pas d'expression explicite, mais se calcule numériquement.